# Pipeline tests: idempotency, upserts, backfill, schema drift, logging and Gold

Part A runs the pipeline on the small files in `test_data/` inside a separate
schema (`world_bank_test`), so the real tables are not touched. Every check prints
PASS/FAIL, and the last cell fails if any check failed.

Every run also refreshes the Gold layer, so each test checks the Gold tables as well:
star schema keys, incremental fact MERGE, aggregate rebuilds and the `silver_to_gold` log rows.

Part B runs read-only checks on the real tables (`world_bank`) after the full and
incremental loads have been run with `run_pipeline`.

In [0]:
%run ./pipeline_utils

In [0]:
TEST_SCHEMA = "world_bank_test"
REAL_SCHEMA = DEFAULT_SCHEMA

test_tables = get_tables("", TEST_SCHEMA)
drop_all_tables(test_tables)      # start every test run from empty tables
create_tables(test_tables)

results = []

def check(name, condition, detail=""):
    results.append((name, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'} - {name} {detail}")

def step_log(run_id, layer):
    rows = (spark.table(test_tables["logs"])
            .filter((F.col("run_id") == run_id) & (F.col("layer") == layer))
            .collect())
    return rows[0] if rows else None

def silver_value(country_key, indicator, year):
    rows = (spark.table(test_tables["silver"])
            .filter((F.col("country_key") == country_key) & (F.col("indicator_code") == indicator) & (F.col("year") == year))
            .select("value").collect())
    return rows[0]["value"] if rows else None

def silver_count():
    return spark.table(test_tables["silver"]).count()

def duplicate_keys(table):
    return spark.table(table).groupBy(*SILVER_KEY).count().filter("count > 1").count()

def gold_steps(run_id):
    """silver_to_gold log rows of one run, keyed by the Gold table they loaded."""
    rows = (spark.table(test_tables["logs"])
            .filter((F.col("run_id") == run_id) & (F.col("layer") == "silver_to_gold"))
            .collect())
    return {r["source"].split(" -> ")[-1]: r for r in rows}

def gold_step(run_id, key):
    return gold_steps(run_id).get(test_tables[key])

def gold_value(country_code, indicator, year, table="fact", column="value"):
    rows = spark.sql(f"""
        SELECT f.{column} FROM {test_tables[table]} f
        JOIN {test_tables['dim_country']} c ON c.country_sk = f.country_sk
        JOIN {test_tables['dim_indicator']} i ON i.indicator_sk = f.indicator_sk
        WHERE c.country_code = '{country_code}' AND i.indicator_code = '{indicator}' AND f.year_key = {year}
    """).collect()
    return rows[0][0] if rows else None

def trend_value(group_name, indicator, year, column="avg_value"):
    rows = spark.sql(f"""
        SELECT t.{column} FROM {test_tables['agg_trends']} t
        JOIN {test_tables['dim_indicator']} i ON i.indicator_sk = t.indicator_sk
        WHERE t.group_name = '{group_name}' AND i.indicator_code = '{indicator}' AND t.year_key = {year}
    """).collect()
    return rows[0][0] if rows else None

def latest_value(country_code, indicator, column="latest_value"):
    rows = spark.sql(f"""
        SELECT l.{column} FROM {test_tables['agg_latest']} l
        JOIN {test_tables['dim_country']} c ON c.country_sk = l.country_sk
        JOIN {test_tables['dim_indicator']} i ON i.indicator_sk = l.indicator_sk
        WHERE c.country_code = '{country_code}' AND i.indicator_code = '{indicator}'
    """).collect()
    return rows[0][0] if rows else None

def gold_orphans(t):
    """Fact / aggregate rows whose key has no dimension row (must be 0)."""
    joins = [("fact", "dim_country", "country_sk"), ("fact", "dim_indicator", "indicator_sk"),
             ("fact", "dim_year", "year_key"), ("agg_yoy", "dim_country", "country_sk"),
             ("agg_yoy", "dim_indicator", "indicator_sk"), ("agg_trends", "dim_indicator", "indicator_sk"),
             ("agg_trends", "dim_year", "year_key"), ("agg_latest", "dim_country", "country_sk"),
             ("agg_latest", "dim_indicator", "indicator_sk")]
    return sum(
        spark.table(t[child]).join(spark.table(t[parent]), col, "left_anti").count()
        for child, parent, col in joins
    )

### Test 1: initial (full) load
34 records: 3 countries x 2 indicators x 5 years, plus 4 records for Niue and
Tokelau, which have an empty country id in the API.

In [0]:
print('add_bronze_metadata' in dir())
print([n for n in dir() if 'metadata' in n.lower() or n.startswith('add_')])

In [0]:
run1, _ = run_pipeline(load_type="full", source_path="test_data/sample_baseline.json",
                       batch_id="test_full", schema=TEST_SCHEMA)

b, s = step_log(run1, "raw_to_bronze"), step_log(run1, "bronze_to_silver")
check("bronze inserted 34 rows", b["rows_inserted"] == 34, f"(got {b['rows_inserted']})")
check("silver inserted 34 rows", s["rows_inserted"] == 34, f"(got {s['rows_inserted']})")
check("silver has 34 rows", silver_count() == 34, f"(got {silver_count()})")
check("entities with empty country id are kept separately",
      spark.table(test_tables["silver"]).filter(F.col("country_key").isin("NAME_NIUE", "NAME_TOKELAU")).count() == 4)
check("every silver row has load_timestamp",
      spark.table(test_tables["silver"]).filter(F.col("load_timestamp").isNull()).count() == 0)

### Test 1b: Gold layer after the initial load
The same run refreshed the star schema: 5 countries (Niue and Tokelau come from the manual
list in `gold_layer`), 3 indicators, 5 years and 34 fact rows.

In [0]:
g = gold_steps(run1)
check("gold: 7 tables + OPTIMIZE logged, all SUCCESS",
      len(g) == 8 and all(r["status"] == "SUCCESS" for r in g.values()),
      f"(got {[(k, r['status']) for k, r in g.items()]})")
check("gold: every table step logs rows_aggregated and duration",
      all(g[test_tables[k]]["rows_aggregated"] is not None and g[test_tables[k]]["duration_seconds"] is not None
          for k in GOLD_KEYS))
check("gold fact inserted 34 rows", gold_step(run1, "fact")["rows_inserted"] == 34,
      f"(got {gold_step(run1, 'fact')['rows_inserted']})")
check("dim_country has 5 rows", spark.table(test_tables["dim_country"]).count() == 5)
check("dim_indicator has 3 rows", spark.table(test_tables["dim_indicator"]).count() == 3)
check("dim_year has 2020-2024", spark.table(test_tables["dim_year"]).count() == 5)
check("Pakistan enriched from countries_metadata.json",
      spark.table(test_tables["dim_country"])
      .filter("country_code = 'PK' AND region = 'South Asia' AND income_level IS NOT NULL").count() == 1)
check("Niue classified as a country in East Asia & Pacific",
      spark.table(test_tables["dim_country"])
      .filter("country_code = 'NAME_NIUE' AND entity_type = 'Country' AND region = 'East Asia & Pacific'").count() == 1)
check("no orphan keys in fact / aggregate tables", gold_orphans(test_tables) == 0)
check("yoy table: one row per reported value (27)", spark.table(test_tables["agg_yoy"]).count() == 27,
      f"(got {spark.table(test_tables['agg_yoy']).count()})")
check("trends: world average electricity 2023 = 98.2",
      abs(trend_value("World", "EG.ELC.ACCS.ZS", 2023) - 98.2) < 1e-9)
check("trends: forest 2024 has 0 reporting countries in South Asia",
      trend_value("South Asia", "AG.LND.FRST.ZS", 2024, "reporting_countries") == 0)
check("latest values: one row per country and indicator with data (6)",
      spark.table(test_tables["agg_latest"]).count() == 6, f"(got {spark.table(test_tables['agg_latest']).count()})")
check("latest values: PK forest = 2023 (2024 not reported yet)",
      latest_value("PK", "AG.LND.FRST.ZS", "latest_year_key") == 2023)

_, aliases = read_country_reference(os.getcwd())
check("ISO3 id ARG and 2-letter id AR map to the same dim_country row",
      {r["alias_code"] for r in aliases.filter(F.col("alias_key").isin("AR", "ARG")).collect()} == {"AR"})

pk_sk = spark.table(test_tables["dim_country"]).filter("country_code = 'PK'").first()["country_sk"]
display(spark.table(test_tables["dim_country"]))

### Test 2: idempotency, re-running the exact same load

In [0]:
silver_before = spark.table(test_tables["silver"]).select(*SILVER_KEY, "value", "load_timestamp").collect()

run2, _ = run_pipeline(load_type="full", source_path="test_data/sample_baseline.json",
                       batch_id="test_full", schema=TEST_SCHEMA)

b, s = step_log(run2, "raw_to_bronze"), step_log(run2, "bronze_to_silver")
silver_after = spark.table(test_tables["silver"]).select(*SILVER_KEY, "value", "load_timestamp").collect()

check("re-run: bronze inserted 0", b["rows_inserted"] == 0, f"(got {b['rows_inserted']})")
check("re-run: silver inserted 0 and updated 0",
      s["rows_inserted"] == 0 and s["rows_updated"] == 0, f"(got {s['rows_inserted']}/{s['rows_updated']})")
check("re-run: silver table unchanged", sorted(silver_before) == sorted(silver_after))
check("no duplicate keys in silver", duplicate_keys(test_tables["silver"]) == 0)

g = gold_steps(run2)
check("re-run: gold fact inserted 0 and updated 0",
      g[test_tables["fact"]]["rows_inserted"] == 0 and g[test_tables["fact"]]["rows_updated"] == 0)
check("re-run: gold dimensions unchanged",
      all(g[test_tables[k]]["rows_inserted"] == 0 and g[test_tables[k]]["rows_updated"] == 0
          for k in ["dim_country", "dim_indicator", "dim_year"]))
check("re-run: aggregate rebuild skipped and no OPTIMIZE",
      all(g[test_tables[k]]["rows_aggregated"] == 0 for k in ["agg_trends", "agg_yoy", "agg_latest"])
      and len(g) == 7)

### Test 3: incremental load with new and revised records (MERGE upsert)
`sample_revisions.json` has 7 records:
3 revised values, 1 value that was missing before (null -> value), 2 unchanged
records and 1 new year (2025).

In [0]:
run3, _ = run_pipeline(load_type="incremental", source_path="test_data/sample_revisions.json",
                       batch_id="test_incremental_1", schema=TEST_SCHEMA)

s = step_log(run3, "bronze_to_silver")
check("incremental: 1 row inserted", s["rows_inserted"] == 1, f"(got {s['rows_inserted']})")
check("incremental: 4 rows updated", s["rows_updated"] == 4, f"(got {s['rows_updated']})")
check("silver has 35 rows", silver_count() == 35, f"(got {silver_count()})")
check("revised value applied (PK electricity 2023 = 95.8)", silver_value("PK", "EG.ELC.ACCS.ZS", 2023) == 95.8)
check("newly reported value applied (PK forest 2024 = 4.62)", silver_value("PK", "AG.LND.FRST.ZS", 2024) == 4.62)

rev = {r["change_type"]: r["count"] for r in
       spark.table(test_tables["revisions"]).groupBy("change_type").count().collect()}
check("revisions table: 3 value_revised + 1 newly_reported",
      rev.get("value_revised") == 3 and rev.get("newly_reported") == 1, f"(got {rev})")

g = gold_step(run3, "fact")
check("gold fact: 1 inserted / 4 updated", (g["rows_inserted"], g["rows_updated"]) == (1, 4),
      f"(got {g['rows_inserted']}/{g['rows_updated']})")
check("gold fact has the revised value (95.8)", gold_value("PK", "EG.ELC.ACCS.ZS", 2023) == 95.8)
check("dim_year extended to 2025", spark.table(test_tables["dim_year"]).agg(F.max("year_key")).first()[0] == 2025)
check("yoy recomputed: PK electricity 2023 = +0.8 vs 2022",
      abs(gold_value("PK", "EG.ELC.ACCS.ZS", 2023, "agg_yoy", "abs_change") - 0.8) < 1e-9)
check("surrogate key of PK did not change",
      spark.table(test_tables["dim_country"]).filter("country_code = 'PK'").first()["country_sk"] == pk_sk)
check("latest values: PK forest moved to the newly reported 2024 value (4.62)",
      (latest_value("PK", "AG.LND.FRST.ZS", "latest_year_key"), latest_value("PK", "AG.LND.FRST.ZS")) == (2024, 4.62))

display(spark.table(test_tables["revisions"]))

In [0]:
# Re-running the same incremental batch must not change anything
run3b, _ = run_pipeline(load_type="incremental", source_path="test_data/sample_revisions.json",
                        batch_id="test_incremental_1", schema=TEST_SCHEMA)
s = step_log(run3b, "bronze_to_silver")
check("incremental re-run: 0 inserted / 0 updated",
      s["rows_inserted"] == 0 and s["rows_updated"] == 0, f"(got {s['rows_inserted']}/{s['rows_updated']})")
check("incremental re-run: no extra revisions", spark.table(test_tables["revisions"]).count() == 4)

g = gold_step(run3b, "fact")
check("incremental re-run: gold fact 0 inserted / 0 updated", (g["rows_inserted"], g["rows_updated"]) == (0, 0))

### Test 4: backfill, re-processing an older batch from Bronze to Silver
Re-running the old `test_full` batch must not overwrite the newer revised values.

In [0]:
run4, _ = run_pipeline(load_type="full", batch_id="test_full", layer="silver", schema=TEST_SCHEMA)
s = step_log(run4, "bronze_to_silver")
check("backfill: 0 inserted / 0 updated",
      s["rows_inserted"] == 0 and s["rows_updated"] == 0, f"(got {s['rows_inserted']}/{s['rows_updated']})")
check("backfill: newer value kept (95.8)", silver_value("PK", "EG.ELC.ACCS.ZS", 2023) == 95.8)

# Full rebuild of Silver from every Bronze batch also gives the same result
run4b, _ = run_pipeline(load_type="full", batch_id="ALL", layer="silver", schema=TEST_SCHEMA)
s = step_log(run4b, "bronze_to_silver")
check("rebuild from ALL batches: 0 inserted / 0 updated",
      s["rows_inserted"] == 0 and s["rows_updated"] == 0, f"(got {s['rows_inserted']}/{s['rows_updated']})")

for r in (run4, run4b):
    g = gold_step(r, "fact")
    check("backfill: gold fact 0 inserted / 0 updated", (g["rows_inserted"], g["rows_updated"]) == (0, 0),
          f"(got {g['rows_inserted']}/{g['rows_updated']})")
check("backfill: gold keeps the newer value (95.8)", gold_value("PK", "EG.ELC.ACCS.ZS", 2023) == 95.8)

### Test 5: schema drift and bad records
`sample_schema_drift.json` has 6 records:
1. a new, unexpected column (`footnote`) -> loaded, extra field kept in `_rescued_data`
2. `value` sent as text `"N/A"` -> quarantined in Bronze->Silver (value_not_numeric)
3. `date` = `"20X5"` -> quarantined in Bronze->Silver (invalid_year)
4. `indicator` object missing -> quarantined in Raw->Bronze (missing_indicator_id)
5. `country` sent as a plain string instead of an object -> quarantined in Raw->Bronze (missing_country)
6. a normal valid record -> loaded

The batch must finish without crashing.

In [0]:
run5, _ = run_pipeline(load_type="incremental", source_path="test_data/sample_schema_drift.json",
                       batch_id="test_drift_1", schema=TEST_SCHEMA)

b, s = step_log(run5, "raw_to_bronze"), step_log(run5, "bronze_to_silver")
check("drift batch finished with SUCCESS", b["status"] == "SUCCESS" and s["status"] == "SUCCESS")
check("raw->bronze: 6 read, 4 inserted, 2 quarantined",
      (b["rows_read"], b["rows_inserted"], b["rows_quarantined"]) == (6, 4, 2),
      f"(got {b['rows_read']}, {b['rows_inserted']}, {b['rows_quarantined']})")
check("bronze->silver: 4 read, 2 inserted, 2 quarantined",
      (s["rows_read"], s["rows_inserted"], s["rows_quarantined"]) == (4, 2, 2),
      f"(got {s['rows_read']}, {s['rows_inserted']}, {s['rows_quarantined']})")
check("new column captured in _rescued_data",
      spark.table(test_tables["bronze"]).filter(F.col("_rescued_data").contains("footnote")).count() == 1)
check("silver has 37 rows", silver_count() == 37, f"(got {silver_count()})")
check("gold fact has 37 rows (same as Silver)", spark.table(test_tables["fact"]).count() == 37)
check("quarantined records never reach Gold", gold_value("IN", "EG.ELC.ACCS.ZS", 2025) is None)

display(spark.table(test_tables["quarantine"]).orderBy("layer", "reason"))
display(spark.table(test_tables["bronze"]).filter(F.col("_rescued_data").isNotNull())
        .select("indicator", "country", "date", "value", "_rescued_data", "batch_id"))

In [0]:
# Re-running the drift batch does not duplicate quarantine rows
quarantine_before = spark.table(test_tables["quarantine"]).count()
run_pipeline(load_type="incremental", source_path="test_data/sample_schema_drift.json",
             batch_id="test_drift_1", schema=TEST_SCHEMA)
check("quarantine not duplicated on re-run", spark.table(test_tables["quarantine"]).count() == quarantine_before)

### Test 6: a failing step is logged as FAILED

In [0]:
try:
    run_pipeline(load_type="incremental", source_path="test_data/file_that_does_not_exist.json",
                 batch_id="test_missing_file", schema=TEST_SCHEMA)
    raised = False
except RuntimeError as e:
    raised = True
    print("Expected error:", e)

failed_logs = (spark.table(test_tables["logs"])
               .filter((F.col("batch_id") == "test_missing_file") & (F.col("status") == "FAILED")).count())
check("missing file raises an error", raised)
check("missing file logged with status FAILED", failed_logs >= 1)

check("no Gold refresh after a failed load",
      spark.table(test_tables["logs"])
      .filter((F.col("batch_id") == "test_missing_file") & (F.col("layer") == "silver_to_gold")).count() == 0)

### Execution log of all test runs

In [0]:
display(spark.table(test_tables["logs"]).orderBy("start_time"))

## Part B: checks on the real tables
Run after loading the real data with `run_pipeline` (full load, then incremental).

In [0]:
real_tables = get_tables("", REAL_SCHEMA)

if spark.catalog.tableExists(real_tables["silver"]):
    check("real silver: no duplicate keys", duplicate_keys(real_tables["silver"]) == 0)
    check("real silver: no null load_timestamp",
          spark.table(real_tables["silver"]).filter(F.col("load_timestamp").isNull()).count() == 0)
    check("real bronze: no null load_timestamp",
          spark.table(real_tables["bronze"]).filter(F.col("load_timestamp").isNull()).count() == 0)

    display(spark.sql(f"""
        SELECT batch_id, load_type, source_file, COUNT(*) AS rows, MIN(load_timestamp) AS loaded_at
        FROM {real_tables['bronze']} GROUP BY batch_id, load_type, source_file ORDER BY loaded_at
    """))
    display(spark.sql(f"""
        SELECT COUNT(*) AS silver_rows,
               SUM(CASE WHEN is_reported THEN 1 ELSE 0 END) AS reported_values,
               COUNT(DISTINCT indicator_code) AS indicators,
               COUNT(DISTINCT country_key) AS countries,
               MIN(year) AS first_year, MAX(year) AS last_year
        FROM {real_tables['silver']}
    """))
    display(spark.table(real_tables["logs"]).orderBy(F.col("start_time").desc()))
else:
    print(f"{real_tables['silver']} does not exist yet, run run_pipeline first.")

In [0]:
if spark.catalog.tableExists(real_tables["fact"]):
    check("real gold: fact rows = silver rows",
          spark.table(real_tables["fact"]).count() == spark.table(real_tables["silver"]).count())
    check("real gold: no duplicate fact keys",
          spark.table(real_tables["fact"]).groupBy(*FACT_KEY).count().filter("count > 1").count() == 0)
    check("real gold: no orphan keys", gold_orphans(real_tables) == 0)
    check("real gold: every entity classified",
          spark.table(real_tables["dim_country"]).filter("entity_type = 'Unclassified'").count() == 0)

    display(spark.sql(" UNION ALL ".join(
        f"SELECT '{real_tables[k]}' AS table_name, COUNT(*) AS row_count FROM {real_tables[k]}" for k in GOLD_KEYS
    )))
else:
    print(f"{real_tables['fact']} does not exist yet, run run_pipeline first.")

In [0]:
failed = [name for name, ok in results if not ok]
print(f"{len(results) - len(failed)} / {len(results)} checks passed")
assert not failed, f"Failed checks: {failed}"